Maya K. Struzak
updated 11/18/2026

# Manuscript statistics

updated 11/18/2026

Statistical tests and summary tables behind the numbers reported in the manuscript, CBP report, and SI.
Reads the derived datasets in `data/derived/` and writes CSV tables to `results/tables/`.

Conventions used throughout:
- Historical period 1991-2000, early future 2021-2050, late future 2051-2090
- A year (or phase-year) is dropped if either the BMP field or its baseline has zero yield. For wheat, the zero-yield years are the fallow year of the wheat-fallow rotation, so they drop out the same way.
- Manure incorporation is compared against `ma-base`, all other BMPs against `base`
- Medians with bootstrapped 95% CIs (10,000 resamples, seed 0); raw p-values

In [ ]:
import os
import numpy as np
import pandas as pd
import pymannkendall as mk
from scipy.stats import mannwhitneyu

DATA = "../../data/derived/"
OUT = "../../results/tables/"
os.makedirs(OUT, exist_ok=True)

HIST = (1991, 2000)
EARLY = (2021, 2050)
LATE = (2051, 2090)
CROPS = ["corn", "soy", "wheat"]
BMPS = ["cc", "gb", "ma", "nm", "nt"]
N_BOOT = 10000
SYN_FILES = {"corn": "synthetic_corn_results.csv", "soy": "synthetic_soy_results.csv", "wheat": "synthetic_wheat_results.csv"}


def load_annual():
    d = pd.read_csv(DATA + "all_annual_results.csv")
    d = d[d.crop.isin(CROPS)]
    hist = (d.scenario == "reference period") & d.year.between(*HIST)
    fut = (d.scenario != "reference period") & d.year.between(EARLY[0], LATE[1])
    d = d[hist | fut].replace([np.inf, -np.inf], np.nan)

    d["fail"] = (d.YLDG + d.YLDF) == 0
    d["ref"] = np.where(d.bmp.isin(["ma", "ma-base"]), "ma-base", "base")
    ref = d[d.bmp == d.ref][["scenario", "crop", "region", "year", "ref", "fail"]].rename(columns={"fail": "ref_fail"})
    d = d.merge(ref, on=["scenario", "crop", "region", "year", "ref"], how="left")
    d = d[~d.fail & ~d.ref_fail]

    d["period"] = "hist"
    for sc, tag in [("RCP 45", "45"), ("RCP 85", "85")]:
        d.loc[(d.scenario == sc) & d.year.between(*EARLY), "period"] = tag + "_early"
        d.loc[(d.scenario == sc) & d.year.between(*LATE), "period"] = tag + "_late"
    return d.drop(columns=["fail", "ref", "ref_fail"])


def load_phase():
    d = pd.read_csv(DATA + "phase_re_dataset.csv")
    hist = (d.scenario == "historical") & d.phase_year.between(*HIST)
    fut = (d.scenario != "historical") & d.phase_year.between(EARLY[0], LATE[1])
    d = d[hist | fut]
    return d[(d.baseline_crop_state == "cropped") & (d.bmp_crop_state == "cropped")]

## Annual

### Trends
Baseline loads are pooled across crops (median per year) for each region. Removal efficiency trends are per BMP, region, and crop.

In [ ]:
d = load_annual()
fut = d[d.period != "hist"]

rows = []
for m in ["TN", "TP", "MUSL"]:
    for (sc, rg), g in fut[fut.bmp == "base"].groupby(["scenario", "region"]):
        s = g.groupby("year")[m].median()
        r = mk.original_test(s.values)
        rows.append([m, sc, rg, len(s), r.slope, r.p, r.trend])
pd.DataFrame(rows, columns=["metric", "scenario", "region", "n_years", "slope_per_yr", "p", "trend"]).to_csv(OUT + "annual_baseline_trends.csv", index=False)

rows = []
for m in ["RE_TN", "RE_TP", "RE_MUSL"]:
    t = fut[fut.bmp.isin(BMPS)].dropna(subset=[m])
    for (sc, bm, rg, cr), g in t.groupby(["scenario", "bmp", "region", "crop"]):
        r = mk.original_test(g.sort_values("year")[m].values)
        rows.append([m, sc, bm, rg, cr, len(g), r.slope, r.slope * 10, r.p, r.trend])
pd.DataFrame(rows, columns=["metric", "scenario", "bmp", "region", "crop", "n_years", "slope_pp_yr", "slope_pp_decade", "p", "trend"]).to_csv(OUT + "annual_re_trends.csv", index=False)

### Period comparisons
Mann-Whitney U between periods for baseline loads and removal efficiencies, per region and crop.

In [ ]:
d = load_annual()
pairs = [("hist", "45_early"), ("hist", "45_late"), ("hist", "85_early"), ("hist", "85_late"), ("45_early", "45_late"), ("85_early", "85_late")]

rows = []
for kind, metrics, bmps in [("baseline", ["TN", "TP", "MUSL"], ["base"]), ("re", ["RE_TN", "RE_TP", "RE_MUSL"], BMPS)]:
    for m in metrics:
        sub = d[d.bmp.isin(bmps)].dropna(subset=[m])
        for (bm, rg, cr), g in sub.groupby(["bmp", "region", "crop"]):
            for a, b in pairs:
                x = g[g.period == a][m]
                y = g[g.period == b][m]
                if len(x) < 5 or len(y) < 5:
                    continue
                rows.append([kind, m, bm, rg, cr, a, b, len(x), len(y), x.median(), y.median(), y.median() - x.median(), mannwhitneyu(x, y).pvalue])

cols = ["kind", "metric", "bmp", "region", "crop", "period_a", "period_b", "n_a", "n_b", "median_a", "median_b", "diff", "p"]
pd.DataFrame(rows, columns=cols).to_csv(OUT + "annual_period_tests.csv", index=False)

### Load gap
Late-century (2051-2090) TN export from each field compared with its own historical load and with the historical untreated baseline.

In [ ]:
d = load_annual()
hist = d[d.period == "hist"]

rows = []
for sc in ["RCP 45", "RCP 85"]:
    late = d[(d.scenario == sc) & d.year.between(*LATE)]
    for cr in CROPS:
        h = hist[hist.crop == cr]
        l = late[late.crop == cr]
        for bm in ["base"] + BMPS:
            hv = h[h.bmp == bm].TN.median()
            lv = l[l.bmp == bm].TN.median()
            if np.isnan(lv):
                continue
            ref = h[h.bmp == ("ma-base" if bm == "ma" else "base")].TN.median()
            rows.append([sc, cr, bm, hv, lv, lv / hv, 100 * (lv / ref - 1)])

cols = ["scenario", "crop", "bmp", "hist_TN", "late_TN", "late_over_hist_own", "pct_vs_hist_untreated"]
pd.DataFrame(rows, columns=cols).round(3).to_csv(OUT + "annual_load_gap.csv", index=False)

### Crop failure
Share of corn and soy years with zero yield, and the effect of the filter on median TN removal efficiency. Wheat is left out because its zero-yield years are the fallow year of the rotation.

In [ ]:
a = pd.read_csv(DATA + "all_annual_results.csv")
a = a[a.crop.isin(["corn", "soy"])]
hist = (a.scenario == "reference period") & a.year.between(*HIST)
fut = (a.scenario != "reference period") & a.year.between(EARLY[0], LATE[1])
a = a[hist | fut].replace([np.inf, -np.inf], np.nan)
a["period"] = np.where(a.scenario == "reference period", "hist", np.where(a.year <= EARLY[1], "early", "late"))
a["failed"] = (a.YLDG + a.YLDF) == 0

summary = (a.groupby(["scenario", "period", "crop", "region", "bmp"])
           .agg(n_years=("failed", "size"), n_failed=("failed", "sum")).reset_index())
summary["pct_failed"] = (100 * summary.n_failed / summary.n_years).round(1)
summary.to_csv(OUT + "crop_failure_summary.csv", index=False)

d = load_annual()
rows = []
for (cr, bm), g in a[a.bmp.isin(BMPS)].groupby(["crop", "bmp"]):
    kept = d[(d.crop == cr) & (d.bmp == bm)]
    rows.append([cr, bm, g.RE_TN.median(), kept.RE_TN.median(), kept.RE_TN.median() - g.RE_TN.median()])
pd.DataFrame(rows, columns=["crop", "bmp", "median_re_unfiltered", "median_re_filtered", "diff_pp"]).round(2).to_csv(OUT + "crop_failure_effect.csv", index=False)

### Removal pathways
Reduction in each load component (median of annual reductions, and reduction in total mass), the soluble share of baseline TN and TP, and the share of mass removed through soluble and sediment-bound pathways.

In [ ]:
d = load_annual()
key = ["scenario", "crop", "region", "year"]
comps = {"MUSL": "sediment", "YN": "sed_N", "QN": "sol_N", "YP": "sed_P", "QP": "sol_P", "TN": "TN", "TP": "TP"}
base = d[d.bmp == "base"].set_index(key)[list(comps)]
mabase = d[d.bmp == "ma-base"].set_index(key)[list(comps)]

rows = []
shares = []
for bm in BMPS:
    j = d[d.bmp == bm].set_index(key)[list(comps)].join(mabase if bm == "ma" else base, rsuffix="_b", how="inner")
    for c, name in comps.items():
        median_red = 100 * (1 - j[c] / j[c + "_b"].replace(0, np.nan)).median()
        mass_red = 100 * (1 - j[c].sum() / j[c + "_b"].sum())
        rows.append([bm, name, median_red, mass_red])
    rm = {c: max((j[c + "_b"] - j[c]).sum(), 0) for c in ["QN", "YN", "QP", "YP"]}
    sol_n = 100 * rm["QN"] / (rm["QN"] + rm["YN"])
    sol_p = 100 * rm["QP"] / (rm["QP"] + rm["YP"])
    shares.append([bm, sol_n, 100 - sol_n, sol_p, 100 - sol_p])

pd.DataFrame(rows, columns=["bmp", "component", "median_annual_reduction_pct", "mass_reduction_pct"]).round(1).to_csv(OUT + "removal_by_component.csv", index=False)
pd.DataFrame(shares, columns=["bmp", "sol_N_pct", "sed_N_pct", "sol_P_pct", "sed_P_pct"]).round(1).to_csv(OUT + "removal_pathway_shares.csv", index=False)

b = d[(d.bmp == "base") & (d.TN > 0) & (d.TP > 0)]
rows = []
for rg, g in [("all", b)] + list(b.groupby("region")):
    rows.append([rg, 100 * (g.QN / g.TN).median(), 100 * g.QN.sum() / g.TN.sum(), 100 * (g.QP / g.TP).median(), 100 * g.QP.sum() / g.TP.sum()])
cols = ["region", "sol_N_share_median_pct", "sol_N_share_mass_pct", "sol_P_share_median_pct", "sol_P_share_mass_pct"]
pd.DataFrame(rows, columns=cols).round(2).to_csv(OUT + "soluble_fractions.csv", index=False)

## Crop-cycle phase

### Efficiency by phase
Median TN removal efficiency by BMP, crop, and phase (all periods), plus the post-harvest minus pre-plant difference within each year.

In [ ]:
d = load_phase()
d = d[np.isfinite(d.re_TN)]
rng = np.random.default_rng(0)

rows = []
for (bm, cr, ph), g in d.groupby(["bmp", "crop", "phase"]):
    v = g.re_TN.values
    draws = np.median(v[rng.integers(0, len(v), (N_BOOT, len(v)))], axis=1)
    rows.append([bm, cr, ph, len(v), np.median(v), np.percentile(draws, 2.5), np.percentile(draws, 97.5)])

p = d.pivot_table(index=["crop", "region", "scenario", "bmp", "phase_year"], columns="phase", values="re_TN").reset_index()
p["diff"] = p.postharvest - p.preplant
for (bm, cr), g in p.groupby(["bmp", "crop"]):
    v = g["diff"].dropna().values
    if len(v) == 0:
        continue
    draws = np.median(v[rng.integers(0, len(v), (N_BOOT, len(v)))], axis=1)
    rows.append([bm, cr, "pre_to_post", len(v), np.median(v), np.percentile(draws, 2.5), np.percentile(draws, 97.5)])

pd.DataFrame(rows, columns=["bmp", "crop", "phase", "n", "median_re", "ci_lo", "ci_hi"]).round(2).to_csv(OUT + "phase_efficiency.csv", index=False)

s = p.set_index(["crop", "region", "scenario", "bmp", "phase_year"])[["preplant", "midseason", "postharvest"]].dropna()
spread = s.max(axis=1) - s.min(axis=1)
pd.DataFrame({"median_pp": [spread.median()], "q25": [spread.quantile(0.25)], "q75": [spread.quantile(0.75)], "n": [len(spread)]}).round(1).to_csv(OUT + "phase_spread.csv", index=False)

### Change under future climate
Regions pooled. Mann-Kendall on the yearly median over 2021-2090, and Mann-Whitney between historical and late century (Figure 4) and between early and late future.

In [ ]:
d = load_phase()
d = d[np.isfinite(d.re_TN)]
hist = d[d.scenario == "historical"]

rows = []
for (sc, bm, cr, ph), g in d[d.scenario != "historical"].groupby(["scenario", "bmp", "crop", "phase"]):
    s = g.groupby("phase_year").re_TN.median()
    r = mk.original_test(s.values)
    e = g[g.phase_year.between(*EARLY)].re_TN
    l = g[g.phase_year.between(*LATE)].re_TN
    h = hist[(hist.bmp == bm) & (hist.crop == cr) & (hist.phase == ph)].re_TN
    p_el = mannwhitneyu(e, l).pvalue if len(e) and len(l) else np.nan
    p_hl = mannwhitneyu(h, l).pvalue if len(h) and len(l) else np.nan
    rows.append([sc, bm, cr, ph, len(s), r.slope * 10, r.p, len(h), h.median(), e.median(), l.median(),
                 l.median() - e.median(), p_el, l.median() - h.median(), p_hl])

cols = ["scenario", "bmp", "crop", "phase", "n_years", "slope_pp_decade", "p_trend", "n_hist", "median_hist", "median_early", "median_late",
        "diff_early_late_pp", "p_early_late", "diff_hist_late_pp", "p_hist_late"]
pd.DataFrame(rows, columns=cols).round(3).to_csv(OUT + "phase_trends.csv", index=False)

### Load redistribution
Share of the annual baseline TN load in each phase.

In [ ]:
d = load_phase()
b = d.drop_duplicates(["crop", "region", "scenario", "phase_year", "phase"]).copy()
tot = b.groupby(["crop", "region", "scenario", "phase_year"]).baseline_TN.transform("sum")
b["share"] = 100 * b.baseline_TN / tot.replace(0, np.nan)

rows = []
for (sc, cr, ph), g in b[b.scenario != "historical"].groupby(["scenario", "crop", "phase"]):
    h = b[(b.scenario == "historical") & (b.crop == cr) & (b.phase == ph)].share.median()
    e = g[g.phase_year.between(*EARLY)].share.median()
    l = g[g.phase_year.between(*LATE)].share.median()
    rows.append([sc, cr, ph, h, e, l, l - e, l - h])

cols = ["scenario", "crop", "phase", "share_hist", "share_early", "share_late", "change_early_late_pp", "change_hist_late_pp"]
pd.DataFrame(rows, columns=cols).round(1).to_csv(OUT + "phase_load_share.csv", index=False)

### Nutrient management soil pools
Monthly soil NO3, NH3, and labile P under nutrient management relative to the baseline field, all periods.

In [ ]:
d = load_phase()
n = d[d.bmp == "nm"]

rows = []
for cr, g in n.groupby("crop"):
    for c in ["ZNO3", "ZNH3", "ZPML"]:
        r = g["bmp_" + c] / g["baseline_" + c].replace(0, np.nan)
        rows.append([cr, c, r.notna().sum(), 100 * (r.median() - 1)])
pd.DataFrame(rows, columns=["crop", "pool", "n", "median_pct_change"]).round(1).to_csv(OUT + "nm_soil_pools.csv", index=False)

## Synthetic storm experiment

### Undefined removal efficiency
Removal efficiency is undefined when the baseline produces no TN load from the storm.

In [ ]:
s = pd.concat([pd.read_csv(DATA + f).assign(crop=c) for c, f in SYN_FILES.items()])
t = s[s.bmp.isin(BMPS)].copy()
t["undef"] = t.RE_TN.isna()
t["warming"] = np.where(t.scenario == "baseline", "unperturbed", "warmed")

rows = []
for cr, g in t.groupby("crop"):
    rows.append([cr, "all", len(g), g.undef.sum()])
    for col in ["window", "region", "return_period", "warming"]:
        for k, gk in g.groupby(col):
            rows.append([cr, ("RP" + str(k)) if col == "return_period" else k, len(gk), gk.undef.sum()])

u = pd.DataFrame(rows, columns=["crop", "stratum", "n_pairs", "n_undefined"])
u["pct_undefined"] = (100 * u.n_undefined / u.n_pairs).round(1)
u.to_csv(OUT + "synthetic_undefined_re.csv", index=False)
print("events with defined RE:", (~t.undef).sum())

### Timing by crop
Median TN removal efficiency for each crop, BMP, and crop-cycle phase across all warming levels, with the range across phases.

In [ ]:
s = pd.concat([pd.read_csv(DATA + f).assign(crop=c) for c, f in SYN_FILES.items()])
t = s[s.bmp.isin(BMPS)].dropna(subset=["RE_TN"])

m = (t.groupby(["crop", "bmp", "window"]).RE_TN.median() * 100).unstack()[["preplant", "midseason", "postharvest"]]
m["range_pp"] = m.max(axis=1) - m.min(axis=1)
m.round(1).to_csv(OUT + "synthetic_timing_by_crop.csv")

### Field state ratios
BMP-to-baseline ratios of surface residue and of sediment yield per unit runoff (an erodibility proxy) at the time of the storm.

In [ ]:
s = pd.concat([pd.read_csv(DATA + f).assign(crop=c) for c, f in SYN_FILES.items()])
key = ["crop", "region", "window", "date_idx", "return_period", "scenario"]
base = s[s.bmp == "base"].set_index(key)
mabase = s[s.bmp == "ma-base"].set_index(key)
t = s[s.bmp.isin(BMPS)].copy()
idx = t.set_index(key).index
for c in ["STL", "MUSL", "Q"]:
    t["b_" + c] = np.where(t.bmp == "ma", mabase.reindex(idx)[c].values, base.reindex(idx)[c].values)

r = t[(t.Q > 0) & (t.b_STL > 0)]
residue = (r.STL / r.b_STL).groupby(r.bmp).median()

e = t[(t.Q > 0) & (t.b_Q > 0) & (t.b_MUSL > 0)]
erod = ((e.MUSL / e.Q ** 1.12) / (e.b_MUSL / e.b_Q ** 1.12)).groupby([e.bmp, e.window]).median().unstack()

pd.concat([residue.rename("residue_ratio"), erod.add_prefix("erod_ratio_")], axis=1).round(3).to_csv(OUT + "synthetic_field_state_ratios.csv")

## Sobol sensitivity
Median first-order and total-order indices by parameter, BMP, and output (single parameters only), for the climate-only and the climate + operations runs.

In [ ]:
for run in ["climate_only", "climate_ops"]:
    s = pd.read_csv(DATA + "sobol_" + run + ".csv")
    s = s[s.crop.isin(CROPS) & ~s.parameter.str.contains(" x ")]
    s.groupby(["parameter", "bmp", "metric"])[["S1", "ST"]].median().round(3).to_csv(OUT + "sobol_median_indices_" + run + ".csv")